In [19]:
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

In [20]:
# -- Loading the video
videoPath = "security_footage.mp4"

# -- loading the model
model = YOLO('yolov8n.pt')


In [ ]:
frame_number = 0
capture = cv2.VideoCapture(videoPath)
track_history = defaultdict(lambda: []) 

while(capture.isOpened()):
    frame_number += 1
    ret, frame = capture.read()
    if ret == False:
        break

    results = model.track(frame, persist=True, classes=0)

    if results[0].boxes.id is not None:
        boxes = results[0].boxes.xywh.cpu()
        track_ids = results[0].boxes.id.int().cpu().tolist()    

        for box, track_id in zip(boxes, track_ids):
            x, y, w, h = box

            center_x = int(x)          
            center_y = int(y)  

            track = track_history[track_id]
            track.append((center_x, center_y))

            if len(track) > 30:
                track.pop(0)

            cv2.circle(frame, (center_x, center_y), 5, (0,0,255), -1)
            points = np.hstack(track).astype(np.int32).reshape((-1,1,2))
            cv2.polylines(frame, [points], isClosed=False, color=(0,255,0), thickness=2)   

    cv2.imshow("Rastro de clientes", frame)

    if cv2.waitKey(1) and 0xFF == ord('q'):
        break

capture.release()
cv2.destroyAllWindows()


0: 384x640 10 persons, 244.1ms
Speed: 3.3ms preprocess, 244.1ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)


error: OpenCV(4.11.0) :-1: error: (-5:Bad argument) in function 'polylines'
> Overload resolution failed:
>  - Argument 'thickness' is required to be an integer
>  - Argument 'thickness' is required to be an integer


In [ ]:
# cv2.imshow("store Tracking", annotated_frame)

# if cv2.waitKey(1) & 0xFF == ord('q'):
#     break

frame_number = 0
capture = cv2.VideoCapture(videoPath)

gridX, gridY= 2, 5
num_images = gridX * gridY  
idx_img = 0

f, ax = plt.subplots(gridY, gridX, figsize=(8,16))
while(capture.isOpened()):
    frame_number += 1
    ret, frame = capture.read()
    if ret == False:
        break

    if frame_number % (1284 // 10) == 0:
        ax[idx_img//gridX, idx_img%gridX].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax[idx_img//gridX, idx_img%gridX].axis(False)
        idx_img += 1
    
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined